
# 15. OptiTrack Re-Do / Repopulation Template (Adapted)

Adapted from `notebooks/tobiasr/data ingest updates/RedoOptitrack.ipynb`
and supporting Natasha optitrack debug notebooks.

This notebook is for safe triage and repopulation planning. It does not write by default.


In [ ]:

import os
from pathlib import Path
import pandas as pd
import datajoint as dj

if Path.cwd().name == "notebooks":
    os.chdir("..")

repo_root = Path.cwd()
cfg = repo_root / "dj_local_conf.json"
if not cfg.exists():
    raise FileNotFoundError("Missing dj_local_conf.json in repository root")

dj.config.load(str(cfg))
dj.conn()

from adamacs.pipeline import subject, session, scan, event
from adamacs.schemas import mocap, virtual_markers_optitrack

ALLOW_DB_WRITES = False
INITIALS = os.environ.get("ADAMACS_INITIALS", "NK")
DATE_FROM = os.environ.get("ADAMACS_DATE_FROM", "2025-01-01")


In [ ]:

keys = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

rows = []
for key in keys:
    rows.append(
        {
            **key,
            "optitrack_events": len(event.Event & key & 'event_type LIKE "%optitrack%"'),
            "motioncapture_rows": len(mocap.MotionCapture & key),
            "rigidmouse_rows": len(virtual_markers_optitrack.RigidMouseTracking & key),
        }
    )

redo_df = pd.DataFrame(rows)
redo_df


In [ ]:

redo_candidates = redo_df[
    (redo_df["optitrack_events"] > 0)
    & (
        (redo_df["motioncapture_rows"] == 0)
        | (redo_df["rigidmouse_rows"] == 0)
    )
]

redo_candidates


In [ ]:

if ALLOW_DB_WRITES:
    raise RuntimeError("Set ALLOW_DB_WRITES manually after explicit approval.")

# Recommended write-enabled execution order:
# 1) mocap.MotionCapture.populate(restriction, ...)
# 2) virtual_markers_optitrack.RigidMouseTracking.populate(restriction, ...)
# 3) pupil_tracking.GazeReconstruction3D.populate(restriction, ...)
